# From JWST / MAST FITS Data to a Beautiful Wallpaper Image

This notebook is for the workflow you actually meant:

**astronomy FITS data → color composite → polished wallpaper export**

It is designed for:
- **uploaded FITS files** you already have
- **optional MAST download mode** for public data
- **JWST** first, but the same ideas often work for HST and other missions if the FITS/WCS are sensible

You do **not** need to train any AI model for this.

## What this notebook does

1. Load raw FITS data from a local folder or from MAST
2. Find usable science image extensions
3. Clean NaNs / infinities and estimate background
4. Reproject all filters onto one common WCS
5. Build a color image from 1, 2, 3, or many filters
6. Apply astronomy-friendly stretches
7. Add a gentle “pretty image” finishing pass
8. Export high-resolution wallpaper crops

## Notes

- If you typed **“JSW”**, I assumed you meant **JWST**.
- This notebook aims for a **beautiful but still sensible** image, not fake prompt-generated art.
- The finishing step is intentionally gentle so the result still reflects the telescope data.

In [ ]:

# If you are in Colab or a fresh notebook environment, run this once.
%pip -q install astroquery astropy reproject scikit-image pillow matplotlib scipy pandas

## Imports

In [ ]:

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re

from PIL import Image, ImageEnhance, ImageFilter

from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import simple_norm, make_lupton_rgb
from astropy.wcs import WCS

from astroquery.mast import Observations
from reproject import reproject_interp

from scipy.ndimage import median_filter
from skimage import exposure

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

plt.rcParams["figure.figsize"] = (8, 8)
plt.rcParams["image.origin"] = "lower"

## Configuration

### Two ways to use this notebook

**Mode A — local uploaded FITS files**
- Put your FITS files into a folder
- Set `DATA_MODE = "local"`

**Mode B — public data from MAST**
- Search by target or proposal
- Set `DATA_MODE = "mast"`

In [ ]:

DATA_MODE = "local"   # "local" or "mast"

LOCAL_DATA_DIR = Path("./fits_input")
EXPLICIT_FITS_FILES = []  # e.g. ["./fits_input/file1.fits"]

MAST_TARGET = "M16"
MAST_RADIUS_DEG = 0.02
MAST_COLLECTION = "JWST"
MAST_MAX_OBS = 15
DOWNLOAD_DIR = Path("./mast_downloads")

PREFERRED_PRODUCT_KEYWORDS = ["i2d", "drz", "drc", "sci"]

WORK_DIR = Path("./work")
OUTPUT_DIR = Path("./output")
WORK_DIR.mkdir(exist_ok=True, parents=True)
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

TARGET_WALLPAPER_SIZE = (3840, 2160)
EXPORT_PHONE_SIZE = (1440, 3120)
BACKGROUND_PERCENTILE_LOW = 1.0
BACKGROUND_PERCENTILE_HIGH = 99.7
STRETCH = "asinh"  # "asinh", "sqrt", "log", "linear"
SATURATION_BOOST = 1.12
CONTRAST_BOOST = 1.08
SHARPEN_RADIUS = 1.2
SHARPEN_AMOUNT = 1.25

MANUAL_CHANNEL_HINTS = {}

## Helper functions

In [ ]:

def list_fits_files(folder: Path):
    if not folder.exists():
        return []
    files = []
    for pat in ["*.fits", "*.fit", "*.fts", "*.fits.gz"]:
        files.extend(folder.rglob(pat))
    return sorted(set(files))


def choose_science_hdu(hdul):
    candidates = []
    for idx, hdu in enumerate(hdul):
        data = getattr(hdu, "data", None)
        if data is None or not isinstance(data, np.ndarray) or data.ndim < 2:
            continue
        name = (getattr(hdu, "name", "") or "").upper()
        header = hdu.header
        score = 0
        if name == "SCI":
            score += 100
        if idx == 0:
            score += 10
        if data.ndim == 2:
            score += 20
        if "WCSAXES" in header or "CTYPE1" in header:
            score += 30
        if data.shape[0] > 128 and data.shape[1] > 128:
            score += 20
        candidates.append((score, idx, hdu))
    if not candidates:
        return None, None, None
    candidates.sort(reverse=True, key=lambda x: x[0])
    _, idx, hdu = candidates[0]
    return idx, hdu.data, hdu.header


def robust_clean(data):
    data = np.array(data, dtype=np.float32)
    bad = ~np.isfinite(data)
    if bad.any():
        med = np.nanmedian(data[~bad]) if (~bad).any() else 0.0
        data[bad] = med
    return data


def background_subtract(data, sigma=3.0, maxiters=10):
    mean, median, std = sigma_clipped_stats(data, sigma=sigma, maxiters=maxiters)
    return data - median, {"mean": float(mean), "median": float(median), "std": float(std)}


def remove_hot_pixels(data, size=3):
    med = median_filter(data, size=size)
    resid = data - med
    _, _, std = sigma_clipped_stats(resid, sigma=3.0)
    mask = resid > 8 * std
    out = data.copy()
    out[mask] = med[mask]
    return out


def normalize_channel(data, low=1.0, high=99.7, stretch="asinh"):
    x = np.array(data, dtype=np.float32)
    lo = np.nanpercentile(x, low)
    hi = np.nanpercentile(x, high)
    if hi <= lo:
        hi = lo + 1e-6
    x = np.clip((x - lo) / (hi - lo), 0, 1)
    if stretch == "sqrt":
        x = np.sqrt(x)
    elif stretch == "log":
        x = np.log1p(1000 * x) / np.log1p(1000)
    elif stretch == "asinh":
        a = 8.0
        x = np.arcsinh(a * x) / np.arcsinh(a)
    elif stretch != "linear":
        raise ValueError(f"Unknown stretch: {stretch}")
    return np.clip(x, 0, 1)


def detect_filter_name(header, path=None):
    keys = ["FILTER", "PUPIL", "FILTNAM1", "FILTNAM2", "FILTER1", "FILTER2"]
    vals = []
    for k in keys:
        if k in header and str(header[k]).strip():
            vals.append(str(header[k]).strip().lower())
    if path is not None:
        stem = Path(path).stem.lower()
        vals.extend(re.findall(r"f\d{3}[wmn]", stem))
    vals = [v for v in vals if v not in {"clear", "none", "nan"}]
    if vals:
        for v in vals:
            if re.match(r"f\d{3}[wmn]", v):
                return v
        return vals[0]
    return Path(path).stem.lower() if path is not None else "unknown"


def wavelength_key(name):
    m = re.search(r"f(\d{3})([wmn])", str(name).lower())
    return int(m.group(1)) if m else 9999


def auto_assign_rgb(filter_names):
    ordered = sorted(filter_names, key=wavelength_key)
    n = len(ordered)
    if n == 1:
        return {"R": ordered[0], "G": ordered[0], "B": ordered[0]}
    if n == 2:
        return {"R": ordered[1], "G": ordered[0], "B": ordered[0]}
    if n == 3:
        return {"B": ordered[0], "G": ordered[1], "R": ordered[2]}
    return {"B": ordered[0], "G": ordered[n // 2], "R": ordered[-1]}


def crop_to_aspect(img, target_size):
    target_w, target_h = target_size
    target_ratio = target_w / target_h
    w, h = img.size
    src_ratio = w / h
    if src_ratio > target_ratio:
        new_w = int(h * target_ratio)
        left = (w - new_w) // 2
        box = (left, 0, left + new_w, h)
    else:
        new_h = int(w / target_ratio)
        top = (h - new_h) // 2
        box = (0, top, w, top + new_h)
    return img.crop(box).resize((target_w, target_h), Image.Resampling.LANCZOS)


def soft_finish(rgb_uint8, saturation=1.12, contrast=1.08, sharpen_radius=1.2, sharpen_amount=1.25):
    img = Image.fromarray(rgb_uint8)
    img = ImageEnhance.Color(img).enhance(saturation)
    img = ImageEnhance.Contrast(img).enhance(contrast)
    img = img.filter(ImageFilter.UnsharpMask(radius=sharpen_radius,
                                             percent=int(100 * sharpen_amount),
                                             threshold=2))
    return img


def show_image(img, title=None, figsize=(10, 10)):
    plt.figure(figsize=figsize)
    plt.imshow(np.asarray(img) if isinstance(img, Image.Image) else img)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def save_image_versions(img, base_name="astronomy_render"):
    paths = {}
    full_path = OUTPUT_DIR / f"{base_name}_full.png"
    img.save(full_path)
    paths["full"] = full_path
    desktop = crop_to_aspect(img, TARGET_WALLPAPER_SIZE)
    desktop_path = OUTPUT_DIR / f"{base_name}_desktop_4k.png"
    desktop.save(desktop_path)
    paths["desktop_4k"] = desktop_path
    phone = crop_to_aspect(img, EXPORT_PHONE_SIZE)
    phone_path = OUTPUT_DIR / f"{base_name}_phone.png"
    phone.save(phone_path)
    paths["phone"] = phone_path
    return paths

## Step 1 — Find data

In [ ]:

def find_local_files():
    if EXPLICIT_FITS_FILES:
        files = [Path(x) for x in EXPLICIT_FITS_FILES]
    else:
        files = list_fits_files(LOCAL_DATA_DIR)
    return [f for f in files if f.exists()]


def search_and_download_mast(target=MAST_TARGET, radius_deg=MAST_RADIUS_DEG,
                             collection=MAST_COLLECTION, max_obs=MAST_MAX_OBS,
                             download_dir=DOWNLOAD_DIR):
    download_dir.mkdir(exist_ok=True, parents=True)
    print(f"Searching MAST for target={target!r}, collection={collection!r} ...")
    obs = Observations.query_object(target, radius=f"{radius_deg} deg")
    if len(obs) == 0:
        raise RuntimeError("No observations found.")
    df = obs.to_pandas()
    if "obs_collection" in df.columns:
        df = df[df["obs_collection"].astype(str).str.upper() == str(collection).upper()]
    if len(df) == 0:
        raise RuntimeError(f"No observations left after filtering to collection={collection!r}")
    df = df.head(max_obs)
    print(f"Found {len(df)} matching observations")
    products = Observations.get_product_list(obs[np.isin(obs['obsid'], df['obsid'].tolist())])
    pdf = products.to_pandas()
    keep = pd.Series(False, index=pdf.index)
    for col in ["productFilename", "description", "productType", "dataproduct_type"]:
        if col in pdf.columns:
            s = pdf[col].astype(str).str.lower()
            for kw in PREFERRED_PRODUCT_KEYWORDS:
                keep = keep | s.str.contains(kw, na=False)
    if "productFilename" in pdf.columns:
        keep = keep | pdf["productFilename"].astype(str).str.lower().str.endswith(".fits")
    pdf = pdf[keep].copy()
    if len(pdf) == 0:
        raise RuntimeError("No suitable FITS-like products found after heuristic filtering.")
    print(f"Downloading {len(pdf)} products...")
    downloaded = Observations.download_products(
        products[np.isin(products['obsID'], pdf['obsID'].tolist())],
        download_dir=str(download_dir),
        cache=True,
        mrp_only=False
    )
    ddf = downloaded.to_pandas()
    local_paths = []
    for col in ["Local Path", "Local_Path", "local_path"]:
        if col in ddf.columns:
            local_paths.extend([Path(p) for p in ddf[col].dropna().tolist()])
    out = []
    for p in local_paths:
        sp = str(p).lower()
        if p.exists() and (sp.endswith(".fits") or sp.endswith(".fits.gz") or p.suffix.lower() in {".fits", ".gz"}):
            out.append(p)
    return sorted(set(out))


if DATA_MODE == "local":
    fits_files = find_local_files()
else:
    fits_files = search_and_download_mast()

print(f"Total FITS files found: {len(fits_files)}")
for f in fits_files[:20]:
    print(" -", f)

## Step 2 — Inspect files and extract science images

In [ ]:

records = []
for path in fits_files:
    try:
        with fits.open(path) as hdul:
            idx, data, header = choose_science_hdu(hdul)
            if data is None:
                print(f"Skipping {path.name}: no usable 2D image HDU found")
                continue
            filt = detect_filter_name(header, path)
            records.append({
                "path": str(path),
                "hdu_index": idx,
                "filter_name": filt,
                "shape": tuple(data.shape),
                "header": header,
            })
    except Exception as e:
        print(f"Skipping {path.name}: {e}")

catalog = pd.DataFrame(records)
catalog

In [ ]:

if len(catalog) == 0:
    raise RuntimeError("No usable science images were found.")

## Preview one image per filter

In [ ]:

def load_record_image(rec):
    with fits.open(rec["path"]) as hdul:
        data = robust_clean(hdul[rec["hdu_index"]].data)
    return data

unique_filters = catalog["filter_name"].dropna().unique().tolist()
preview_rows = []
for filt in unique_filters:
    rec = catalog[catalog["filter_name"] == filt].iloc[0]
    data = load_record_image(rec)
    data, stats = background_subtract(data)
    data = remove_hot_pixels(data)
    preview_rows.append((filt, data, stats))

n = len(preview_rows)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]

for ax, (filt, data, stats) in zip(axes, preview_rows):
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, norm=norm, cmap="gray")
    ax.set_title(f"{filt}\nmedian={stats['median']:.3g}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Step 3 — Pick one image per filter

In [ ]:

catalog["n_pix"] = catalog["shape"].apply(lambda s: int(np.prod(s[-2:])))
best_per_filter = (
    catalog.sort_values("n_pix", ascending=False)
           .drop_duplicates("filter_name")
           .reset_index(drop=True)
)
best_per_filter[["filter_name", "path", "hdu_index", "shape", "n_pix"]]

In [ ]:

MANUAL_FILE_OVERRIDE = {
    # "f090w": "/full/path/to/file.fits",
}

In [ ]:

selection = {}
for _, row in best_per_filter.iterrows():
    filt = row["filter_name"]
    path = MANUAL_FILE_OVERRIDE.get(filt, row["path"])
    selection[filt] = {"path": path, "hdu_index": int(row["hdu_index"])}
selection

## Step 4 — Load, clean, and align all filters onto a common WCS

In [ ]:

selected_filters = sorted(selection.keys(), key=wavelength_key)
print("Selected filters:", selected_filters)
reference_filter = selected_filters[0]
print("Reference filter:", reference_filter)

loaded = {}
for filt in selected_filters:
    path = selection[filt]["path"]
    hdu_index = selection[filt]["hdu_index"]
    with fits.open(path) as hdul:
        data = robust_clean(hdul[hdu_index].data)
        header = hdul[hdu_index].header
    data, stats = background_subtract(data)
    data = remove_hot_pixels(data)
    loaded[filt] = {
        "data": data,
        "header": header,
        "wcs": WCS(header),
        "stats": stats,
        "path": path,
    }

ref = loaded[reference_filter]
ref_header = ref["header"]
ref_shape = ref["data"].shape

aligned = {}
for filt, item in loaded.items():
    if filt == reference_filter:
        aligned[filt] = item["data"]
        continue
    try:
        reprojected, footprint = reproject_interp((item["data"], item["wcs"]),
                                                  ref_header,
                                                  shape_out=ref_shape)
        aligned[filt] = robust_clean(reprojected)
        print(f"Aligned {filt} -> {reference_filter}")
    except Exception as e:
        print(f"Could not align {filt}: {e}")

list(aligned.keys())

## Check aligned previews

In [ ]:

n = len(aligned)
fig, axes = plt.subplots(1, n, figsize=(5*n, 5))
if n == 1:
    axes = [axes]
for ax, filt in zip(axes, aligned.keys()):
    data = aligned[filt]
    norm = simple_norm(data, stretch=STRETCH, percent=99.5)
    ax.imshow(data, norm=norm, cmap="gray")
    ax.set_title(filt)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Step 5 — Choose color mapping

In [ ]:

auto_rgb = auto_assign_rgb(list(aligned.keys()))
print("Automatic RGB assignment:", auto_rgb)
rgb_assignment = auto_rgb.copy()
for key, ch in MANUAL_CHANNEL_HINTS.items():
    key = key.lower()
    ch = ch.upper()
    if key in aligned and ch in {"R", "G", "B"}:
        rgb_assignment[ch] = key
print("Final RGB assignment:", rgb_assignment)

## Step 6 — Build the RGB composite

In [ ]:

def prepare_rgb_channels(aligned_data, rgb_assignment,
                         low=BACKGROUND_PERCENTILE_LOW,
                         high=BACKGROUND_PERCENTILE_HIGH,
                         stretch=STRETCH):
    r = normalize_channel(aligned_data[rgb_assignment["R"]], low=low, high=high, stretch=stretch)
    g = normalize_channel(aligned_data[rgb_assignment["G"]], low=low, high=high, stretch=stretch)
    b = normalize_channel(aligned_data[rgb_assignment["B"]], low=low, high=high, stretch=stretch)
    return r, g, b

r, g, b = prepare_rgb_channels(aligned, rgb_assignment)
rgb_float = np.dstack([r, g, b])

plt.figure(figsize=(10, 10))
plt.imshow(rgb_float)
plt.title("Initial RGB composite")
plt.axis("off")
plt.show()

### Optional: Lupton RGB rendering

In [ ]:

r_raw = aligned[rgb_assignment["R"]]
g_raw = aligned[rgb_assignment["G"]]
b_raw = aligned[rgb_assignment["B"]]

try:
    lupton_rgb = make_lupton_rgb(r_raw, g_raw, b_raw, stretch=5, Q=8)
    plt.figure(figsize=(10, 10))
    plt.imshow(lupton_rgb)
    plt.title("Lupton RGB preview")
    plt.axis("off")
    plt.show()
except Exception as e:
    print("Lupton RGB preview failed:", e)
    lupton_rgb = None

## Step 7 — Choose which base render to continue with

In [ ]:

USE_LUPTON_BASE = False

if USE_LUPTON_BASE and lupton_rgb is not None:
    base_rgb_uint8 = np.asarray(lupton_rgb).astype(np.uint8)
else:
    base_rgb_uint8 = np.clip(rgb_float * 255, 0, 255).astype(np.uint8)

show_image(base_rgb_uint8, "Chosen base render")

## Step 8 — Pretty-image finishing pass

In [ ]:

def tone_finish(rgb_uint8):
    x = np.asarray(rgb_uint8).astype(np.float32) / 255.0
    x = np.clip(x, 0, 1)
    x = exposure.adjust_gamma(x, gamma=0.95)
    lum = x.mean(axis=2)
    p2, p98 = np.percentile(lum, (1, 99.5))
    if p98 > p2:
        x = exposure.rescale_intensity(x, in_range=(p2, p98), out_range=(0, 1))
    return (np.clip(x, 0, 1) * 255).astype(np.uint8)

toned = tone_finish(base_rgb_uint8)
finished_img = soft_finish(
    toned,
    saturation=SATURATION_BOOST,
    contrast=CONTRAST_BOOST,
    sharpen_radius=SHARPEN_RADIUS,
    sharpen_amount=SHARPEN_AMOUNT
)

show_image(finished_img, "Finished render")

## Before / after comparison

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(base_rgb_uint8)
axes[0].set_title("Base composite")
axes[0].axis("off")

axes[1].imshow(finished_img)
axes[1].set_title("Finished render")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Step 9 — Export wallpaper versions

In [ ]:

exported = save_image_versions(finished_img, base_name="jwst_pretty_render")
exported

In [ ]:

for label, path in exported.items():
    print(label, "->", path)
    show_image(Image.open(path), title=label, figsize=(10, 6) if "desktop" in label else (5, 10))

## Optional: save intermediate products too

In [ ]:

Image.fromarray(base_rgb_uint8).save(OUTPUT_DIR / "jwst_base_composite.png")
Image.fromarray(toned).save(OUTPUT_DIR / "jwst_tone_finished.png")
print("Saved intermediate PNGs in", OUTPUT_DIR.resolve())

## Advanced notes

### When you have more than 3 filters
This notebook maps:
- shortest wavelength → blue
- middle wavelength → green
- longest wavelength → red

For more curated images, you can blend multiple filters per RGB channel.

### When data look noisy or washed out
Try:
- lowering `BACKGROUND_PERCENTILE_LOW`
- lowering `BACKGROUND_PERCENTILE_HIGH`
- switching `STRETCH` between `asinh`, `sqrt`, and `log`
- reducing `SATURATION_BOOST`
- turning off `USE_LUPTON_BASE`

### When colors look wrong
Edit `MANUAL_CHANNEL_HINTS` or directly override `rgb_assignment`.

## Optional cell — custom blend for many filters

In [ ]:

# Example:
# custom_map = {
#     "B": ["f090w", "f115w"],
#     "G": ["f150w", "f200w"],
#     "R": ["f277w", "f356w", "f444w"],
# }

def blend_filters(aligned_data, filter_list, low=BACKGROUND_PERCENTILE_LOW,
                  high=BACKGROUND_PERCENTILE_HIGH, stretch=STRETCH):
    arrs = []
    for f in filter_list:
        if f in aligned_data:
            arrs.append(normalize_channel(aligned_data[f], low=low, high=high, stretch=stretch))
    if not arrs:
        return None
    return np.mean(arrs, axis=0)

# custom_r = blend_filters(aligned, custom_map["R"])
# custom_g = blend_filters(aligned, custom_map["G"])
# custom_b = blend_filters(aligned, custom_map["B"])
# custom_rgb = np.dstack([custom_r, custom_g, custom_b])
# show_image((np.clip(custom_rgb, 0, 1) * 255).astype(np.uint8), "Custom multi-filter blend")

## Optional cell — process a single grayscale FITS into a false-color wallpaper

In [ ]:

def false_color_from_single(data, cmap="inferno", low=1, high=99.7, stretch="asinh"):
    x = normalize_channel(data, low=low, high=high, stretch=stretch)
    cm = plt.get_cmap(cmap)
    rgb = (cm(x)[..., :3] * 255).astype(np.uint8)
    return Image.fromarray(rgb)

# Example:
# only_filter = list(aligned.keys())[0]
# single_img = false_color_from_single(aligned[only_filter], cmap="magma")
# show_image(single_img, f"False-color render: {only_filter}")

## Minimal recipe

1. Open FITS
2. Choose science HDU
3. Background subtract
4. Remove bad pixels
5. Reproject all filters to a common WCS
6. Assign channels by wavelength
7. Stretch with `asinh`
8. Make RGB
9. Apply gentle finishing
10. Export 4K wallpaper